<b><h2 style="color:blue;">IPB - Praksa</h2>
<h4>Grupa 4:</h4>
<h3 style="color: green">Simulacija Habitabilnosti planete Dragica </h3>


Model spaja više faktora i računa verovatnoću na osnovu:
$$H(r, h) = f_T(r,h) \cdot f_P(r,h) \cdot f_E(r,h) \cdot f_{pH}(r,h)$$

Gde su:
* $r$ — horizontalna udaljenost od hidrotermalnog izvora [km]
* $h$ — visina iznad dna okeana [km]
* $f_T$ — faktor temperature 
* $f_P$ — faktor dostupnosti fosfora (Michaelis-Menten kinetika)
* $f_E$ — faktor dostupnosti energije
* $f_{pH}$ — faktor kiselosti

Svi faktori imaju vrednosti u opsegu $[0, 1]$.
kombinaciju $H(r, h) \in [0, 1]$ lokalna verovatnoca nastanjivosti.

In [22]:
# potrebne biblioteke
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, Rectangle
from scipy.ndimage import gaussian_filter

In [23]:
# parametri sistema

# parametri zvezde 
ZVEZDA = {
    "klasa"          : "M3V",
    "T_eff_K"        : 3400, # efektivna temperatura [K]
    "masa_Msun"      : 0.45, # masa izrazena preko mase sun
    "radijus_Rsun"   : 0.42, # radijus -||-
    "metalicnost"    : 0.05, # [Fe/H]
    "luminoznost_Lsun": 0.02, # luminoznost preko lum. sunca
}

# parametri planete
PLANETA = {
    "masa_mEarth"    : 2.4, # masa izrazena preko mase zemlje
    "radijus_Rearth" : 1.25, # radijus -||-
    "gravitacija_g"  : 1.52, # površinska gravitacija -||-
    "orbita_AU"      : 0.20, # velika poluosa [AU]
    "ekscentricitet" : 0.02,
    "pritisak_bar"   : 2.0, # površinski atmosferski pritisak u barima
    "T_povrsina_C"   : -30.0, # površinska temperatura leda u celzijusima
}

# atmosferski sastav
ATM = {
    "CO2_mol_frac"   : 0.870, # molski udeo kad se prebaci u procente 87% 
    "N2_mol_frac"    : 0.100, 
    "Ar_mol_frac"    : 0.018,
    "H2O_mol_frac"   : 0.007,
    "H2_mol_frac"    : 0.003,
    "CH4_ppm"        : 80.0, # delova po milionu u procentima 0.008%
    "SO2_ppm"        : 25.0,
}

# parametri pod ledenog okeana
OKEAN = {
    "dubina_km"      : 40.0, # ukupna dubina okeana [km]
    "pH_bulk"        : 6.5, # pH daleko od termalnih izvora
    "salinitet_pct"  : 1.0, # salinitet [%]
    "T_led_C"        : -1.0, # temperatura na granici led–voda [C]
    "T_dno_C"        : 7.3, # ambijenalna temperatura dna [C]
}

# parametri termalnih izvora
VENT = {
    "T_crni_C"       : 258.0, # temperatura crnog pušača [C]
    "pH_vent"        : 4.5, # pH neposredno kod izvora
    "P_vent_uM"      : 50.0, # koncentracija fosfora kod izvora [mikroM]
    "H2_vent_mM"     : 15.0, # koncentracija H2 kod izvora [mM]
    "H2S_vent_mM"    : 3.0, # koncentracija H2S [mM]
}

# bioloski parametri
BIO = {
    "T_opt_C"        : 10.0, # optimalna temperatura [C]
    "T_sigma_C"      : 15.0, # sirina temp. prozora [C]
    "T_max_C"        : 110.0, # gornja granica [C]
    "T_min_C"        : -15.0, # donja granica [C]
    "Km_P_uM"        : 1.0, # Michaelis konstanta za fosfor [mikroM]
    "Km_E_mM"        : 0.10, # Michaelis konstanta za H2 [mM]
    "pH_opt"         : 6.0, # optimalni pH
    "pH_sigma"       : 1.5, # tolerancija pH
    "P_min_uM"       : 0.1, # minimalna koncentracija P za rast [mikroM]
}

# prostorni parametri modela
GRID = {
    "R_max_km"       : 500,         # maksimalni horizontalni domet [km]
    "Nr"             : 4000,         # broj tačaka u r-pravcu
    "Nh"             : 2500,         # broj tačaka u h-pravcu
}


In [24]:
# pravljenje prostorne mreze
H_ok  = OKEAN["dubina_km"]
R_max = GRID["R_max_km"]
r = np.linspace(0.05, R_max, GRID["Nr"])   # km, mora 0.05 da se izbegne singularitet na r=0
h = np.linspace(0.0,  H_ok,  GRID["Nh"])   # km (0 = dno, H_ok = led)

R, H_grid = np.meshgrid(r, h)              # oblici: (Nh, Nr)
print(R.shape) # istampa u konzolu oblik matrice u ovom slucaju nase mape
print(H_grid.shape)


(2500, 4000)
(2500, 4000)


Pre samog jezgra modela potrebno je popuniti mapu (matricu) koju smo dobili. Sledeci blok kod uzima svaki od definisanih faktora i za pocetak popunjava celu mapu. Prostim jezikom, za svaku koordinatu mape mi joj dodeljujemo vrednost za temperaturu, kolicinu fosfora, energiju, i pH vrednost.

In [25]:
T_background = OKEAN["T_led_C"] + (OKEAN["T_dno_C"] - OKEAN["T_led_C"]) * (1.0 - H_grid / H_ok)
print(np.flipud(T_background)) # dobije se matrica sa svim vrednostima temperature u svakom polju

[[-1.         -1.         -1.         ... -1.         -1.
  -1.        ]
 [-0.99667867 -0.99667867 -0.99667867 ... -0.99667867 -0.99667867
  -0.99667867]
 [-0.99335734 -0.99335734 -0.99335734 ... -0.99335734 -0.99335734
  -0.99335734]
 ...
 [ 7.29335734  7.29335734  7.29335734 ...  7.29335734  7.29335734
   7.29335734]
 [ 7.29667867  7.29667867  7.29667867 ...  7.29667867  7.29667867
   7.29667867]
 [ 7.3         7.3         7.3        ...  7.3         7.3
   7.3       ]]


Kako je kod iznad temperatura u svakoj koordinati na mapi ne racunajuci termalni izvor. Potrebno je sada odraditi korekciju za to.

In [26]:
#hidrotermalni izvor, eksponencijalni pad sa udaljenoscu i visinom
r_T = 4.0   # horizontalni domet [km]
h_T = 2.5   # vertikalni domet plumona [km]

T_plume = (VENT["T_crni_C"] - T_background) * np.exp(-R / r_T) * np.exp(-H_grid / h_T)
T_plume = np.maximum(T_plume, 0.0) # zastitna mera ako vrednosti odu u negativne (nema fizicki smisla da termalni izvor hladi vodu)

T_field = T_background + T_plume   # ukupno temperaturno polje [C]
print(np.flipud(T_field))

[[ -0.99997122  -0.9999721   -0.99997296 ...  -1.          -1.
   -1.        ]
 [ -0.9966497   -0.99665059  -0.99665146 ...  -0.99667867  -0.99667867
   -0.99667867]
 [ -0.99332819  -0.99332909  -0.99332995 ...  -0.99335734  -0.99335734
   -0.99335734]
 ...
 [251.73543489 244.21363196 236.92328476 ...   7.29335734   7.29335734
    7.29335734]
 [253.30557329 245.73555735 238.39848071 ...   7.29667867   7.29667867
    7.29667867]
 [254.88575458 247.2672166  239.883111   ...   7.3          7.3
    7.3       ]]


Isti proces razmisljanja se primenjuje i kod fosfora. Kao izvor je uzeto oslobadjanje iz apatita u bazaltnim stenama kroz hidrotermalne procese. On se zatim transportuje kroz okean u horizontalnom i vertikalnom pravcu, dok do gubitka dolazi u ovom modelu iz 2 efekta prvi je precipitacijom , a druga je apsorpcija zbog gvozdje-oksihidroksid ($FeOOH$) tj. blizu izvora gde ima dosta $Fe^{3+}$.

In [27]:
r_P   = 30.0   # horizontalni transport [km]
h_P   = 18.0   # vertikalni transport [km]

P_transport = np.exp(-R / r_P) * (1.0 - np.exp(-H_grid / 1.5))
P_transport = np.maximum(P_transport, 0.0) # zastitna mera kao kod temp.

# Ca-precipitacija: jaci efekat dalje od kiselih uslova venta -> nece biti tipican pad kao kod gausijana vec ce biti lorencijan koji opada kao x^(-2)
r_Ca  = 80.0
f_Ca  = 1.0 / (1.0 + (R / r_Ca)**2)

# apsorpcija, eksponencijalno opada sa udaljenoscu
r_Fe  = 6.0
f_Fe  = 1.0 - 0.85 * np.exp(-R / r_Fe)

P_field = VENT["P_vent_uM"] * P_transport * f_Ca * f_Fe   # [mikroM]
P_field = np.maximum(P_field, 0.0)
print(P_field)

[[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [8.32112626e-02 9.20361066e-02 1.00597210e-01 ... 7.72540483e-10
  7.68952620e-10 7.65381508e-10]
 [1.65539304e-01 1.83095323e-01 2.00126660e-01 ... 1.53688107e-09
  1.52974342e-09 1.52263910e-09]
 ...
 [7.83961508e+00 8.67103354e+00 9.47760407e+00 ... 7.27836573e-08
  7.24456325e-08 7.21091859e-08]
 [7.83961508e+00 8.67103354e+00 9.47760407e+00 ... 7.27836573e-08
  7.24456325e-08 7.21091859e-08]
 [7.83961508e+00 8.67103354e+00 9.47760407e+00 ... 7.27836573e-08
  7.24456325e-08 7.21091859e-08]]


Dalje bez nekog preteranog detaljisanja sa istom idejom se dobija i kod za energiju i pH. Kod energije je kao glavni izvor uzeta serpentizacija, dok je glavni gubitak mikrobioloska potrosnja.

In [28]:
r_E   = 15.0   # horizontalni domet [km]
h_E   = 12.0   # vertikalni domet [km]

E_field = VENT["H2_vent_mM"] * np.exp(-R / r_E) * (1.0 - np.exp(-H_grid / 2.0))
E_field = np.maximum(E_field, 0.0) # zastitna mera kao kod temp
print(E_field, "\n")

r_pH  = 8.0    # km — oporavak pH od venta
pH_field = OKEAN["pH_bulk"] - (OKEAN["pH_bulk"] - VENT["pH_vent"]) * np.exp(-R / r_pH)

# Korekcija za sniženu rastvorljivost CO₂ pri hladnijem gornjem okeanu
pH_field += 0.4 * (H_grid / H_ok)
print(pH_field)

[[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.19171014e-01 1.18181901e-01 1.17200998e-01 ... 4.05858720e-16
  4.02490114e-16 3.99149468e-16]
 [2.37392086e-01 2.35421744e-01 2.33467756e-01 ... 8.08482235e-16
  8.01771875e-16 7.95117211e-16]
 ...
 [1.49500832e+01 1.48259983e+01 1.47029433e+01 ... 5.09152471e-14
  5.04926532e-14 5.00735668e-14]
 [1.49500832e+01 1.48259983e+01 1.47029433e+01 ... 5.09152471e-14
  5.04926532e-14 5.00735668e-14]
 [1.49500832e+01 1.48259983e+01 1.47029433e+01 ... 5.09152471e-14
  5.04926532e-14 5.00735668e-14]] 

[[4.51246102 4.54327954 4.5736202  ... 6.5        6.5        6.5       ]
 [4.51262108 4.54343961 4.57378026 ... 6.50016006 6.50016006 6.50016006]
 [4.51278115 4.54359967 4.57394032 ... 6.50032013 6.50032013 6.50032013]
 ...
 [4.91214089 4.94295941 4.97330007 ... 6.89967987 6.89967987 6.89967987]
 [4.91230095 4.94311948 4.97346013 ... 6.89983994 6.89983994 6.89983994]
 [4.91246102 4.94327954 4.97

Kada su popunjena sva polja mape za svaki od 4 odabrana faktora sada je vreme da se sve to spoji. Kao sto je vec navedeno svaki habitabilni faktor je broj u opsegu $[0, 1]$. Vrednost 0 znaci potpuno nepogodne uslove a 1 idealne. 

Temperatura ce biti modelovana Gausijanom ( $f(x) = \frac{1}{\sqrt{2\pi \sigma^2}}e^{(-\frac{(x-\mu)^2}{2\sigma^2})}$ ) oko optimalne temperature koja je u nasem slucaju promenljiva T_opt_C te formula postaje $f_T = e^{(-(T\_field - T\_opt\_C) / T\_sigma\_C)^2}$. Izostavljamo prvi deo formule jer nam je potrebna normalizacija tako da max vrednost bude kada se poklope optimalna temperatura i temperatura u matrici, sto je u nasem lsucaju ta maksimalna vrednost od 1. Takodje $f_T$ ce biti 0 ako predje gornju i donju granicu

In [29]:
f_T = np.exp(-((T_field - BIO["T_opt_C"]) / BIO["T_sigma_C"])**2)
f_T[T_field > BIO["T_max_C"]] = 0.0   # gornja granica
f_T[T_field < BIO["T_min_C"]] = 0.0   # donja granica
print(f_T)

[[0.         0.         0.         ... 0.96811926 0.96811926 0.96811926]
 [0.         0.         0.         ... 0.96804204 0.96804204 0.96804204]
 [0.         0.         0.         ... 0.96796474 0.96796474 0.96796474]
 ...
 [0.5844257  0.58442565 0.5844256  ... 0.58442404 0.58442404 0.58442404]
 [0.58423601 0.58423596 0.58423591 ... 0.58423436 0.58423436 0.58423436]
 [0.58404633 0.58404628 0.58404623 ... 0.58404469 0.58404469 0.58404469]]
